## Import

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from lime.lime_tabular import LimeTabularExplainer

if not hasattr(np, 'int'):
    np.int = int

seed = 0

## Dataset

In [32]:
all_times_df = pd.read_csv('predictors_plus_target_calibrated.csv')
all_times_df = all_times_df.drop(['p6177_i0', 'p90004', 'p4079_i0_a0', 'p94_i0_a0', 'p4080_i0_a0', 'p93_i0_a0', 'p20002_i0_a0', 'p6177_i0', 'p30760_i0', 'p30870_i0'], axis=1)
all_times_df = all_times_df.dropna()
print('Number of Individuals ', len(all_times_df))

Number of Individuals  11583


In [33]:
label_encoder = LabelEncoder()
all_times_df['activity_class'] = label_encoder.fit_transform(all_times_df['activity_class'])

all_times_df['p31'] = pd.factorize(all_times_df['p31'])[0]
all_times_df['p2306_i0'] = pd.factorize(all_times_df['p2306_i0'])[0]
all_times_df['p2443_i0'] = pd.factorize(all_times_df['p2443_i0'])[0]

In [34]:
scaler = StandardScaler()
predictors = ['Mean', 'STD', 'skewR', 'kurtR', 'perc95', 'perc5', 'dPerc', 'Peak power', 'PPFd', 'Entropy', 'PF_sum', 'P_sum', 'p31','Steps']
all_times_df[predictors] = scaler.fit_transform(all_times_df[predictors])

In [35]:

X = all_times_df[['Mean', 'STD', 'skewR', 'kurtR', 'perc95', 'perc5', 'dPerc', 'Peak power', 'PPFd', 'Entropy', 'PF_sum', 'P_sum', 'p31','Steps']]
y = all_times_df['activity_class']

In [36]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

In [37]:
rf_classifier = RandomForestClassifier(random_state=seed)
rf_classifier.fit(X_train_full_scaled, y_train_full)

RandomForestClassifier(random_state=0)

## Counterfactuals

In [38]:
def generate_lime_c_counterfactuals(df, target_col, model, numeric_cols, sum_threshold=3.0):
    df_use = df.copy(deep=True)
    X = df_use[numeric_cols].values
    y = model.predict(df_use[numeric_cols])


    explainer = LimeTabularExplainer(
        training_data=X,
        feature_names=numeric_cols,
        class_names=["Active", "Sedentary"],
        mode='classification'
    )

    preds = y
    x_nolabel = df_use[numeric_cols]

    def feasible_sum(i):
        return abs(x_nolabel.iloc[i].sum()) < sum_threshold

    row_idx = None
    idxs_1 = np.where(preds == 1)[0]
    for i in idxs_1:
        if feasible_sum(i):
            row_idx = i
            break
    if row_idx is None:
        idxs_0 = np.where(preds == 0)[0]
        for i in idxs_0:
            if feasible_sum(i):
                row_idx = i
                break

    if row_idx is None:
        print("No suitable row found to flip within sum_threshold =", sum_threshold)
        return None, None
    
    chosen_sum = x_nolabel.iloc[row_idx].sum()
    chosen_pred = preds[row_idx]
    print(f"Picked row {row_idx}, sum={chosen_sum:.2f}, predicted_label={chosen_pred}")

    instance = X[row_idx]
    explanation = explainer.explain_instance(
        data_row=instance,
        predict_fn=model.predict_proba,
        num_features=len(numeric_cols),
        top_labels=1
    )

    exp_map = explanation.local_exp[chosen_pred]
    feature_weights = {}
    for feat_idx, w in exp_map:
        feat_name = numeric_cols[feat_idx]
        feature_weights[feat_name] = w

    if chosen_pred == 1:
        candidate_feats = [(f, w) for f, w in feature_weights.items() if w > 0]
        if not candidate_feats:
            print("No positively contributing features found")
            return instance, explanation
        top_feature, top_weight = max(candidate_feats, key=lambda x: x[1])
        flip_direction = 1
    else:
        candidate_feats = [(f, w) for f, w in feature_weights.items() if w < 0]
        if not candidate_feats:
            print("No negatively contributing features found")
            return instance, explanation
        top_feature, top_weight = min(candidate_feats, key=lambda x: x[1])
        flip_direction = -1

    old_instance_df = pd.DataFrame([instance], columns=numeric_cols)
    new_instance = instance.copy()
    feature_idx = numeric_cols.index(top_feature)
    shift_amount = 1.0 * flip_direction
    new_instance[feature_idx] += shift_amount

    new_pred = model.predict(pd.DataFrame([new_instance], columns=numeric_cols))[0]
    print(f"Attempted to flip '{top_feature}' by {shift_amount:.2f}")
    print(f"New predicted label = {new_pred}")

    new_instance_df = pd.DataFrame([new_instance], columns=numeric_cols)
    delta_df = new_instance_df - old_instance_df

    comparison = pd.concat([old_instance_df, new_instance_df, delta_df],
                           keys=["Old", "New", "Delta"], axis=0).T
    print("\n=== Feature Comparison ===")
    print(comparison)

    return new_instance, explanation


In [39]:
german_cf, german_lime_exp = generate_lime_c_counterfactuals(
    df=all_times_df,
    target_col='activity_class',
    model=rf_classifier,
    numeric_cols=predictors,
    sum_threshold=2.0
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Picked row 3, sum=-0.97, predicted_label=1
Attempted to flip 'Mean' by 1.00
New predicted label = 0

=== Feature Comparison ===
                     Old           New Delta
                       0             0     0
Mean       -9.628826e-01  3.711735e-02   1.0
STD        -9.343983e-01 -9.343983e-01   0.0
skewR       1.991178e-09  1.991178e-09   0.0
kurtR      -1.858637e-02 -1.858637e-02   0.0
perc95     -9.352729e-01 -9.352729e-01   0.0
perc5      -5.682144e-02 -5.682144e-02   0.0
dPerc      -9.321542e-01 -9.321542e-01   0.0
Peak power  6.490530e-01  6.490530e-01   0.0
PPFd        4.305577e-01  4.305577e-01   0.0
Entropy     2.700102e-01  2.700102e-01   0.0
PF_sum      2.021761e-01  2.021761e-01   0.0
P_sum       1.976233e-01  1.976233e-01   0.0
p31         8.801298e-01  8.801298e-01   0.0
Steps       2.385588e-01  2.385588e-01   0.0


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
